In [ ]:
!pip install tqdm

In [18]:
import os
import json
from tqdm import tqdm
from pathlib import Path
from collections import defaultdict
import shutil

def convert_coco_to_yolo(json_path, image_dir, output_dir, subset):
    with open(json_path, 'r', encoding='utf-8') as f:
        coco = json.load(f)

    images = {img["id"]: img for img in coco["images"]}
    categories = {cat["id"]: i for i, cat in enumerate(coco["categories"])}  # id 映射到 0-based index

    label_dir = os.path.join(output_dir, subset, 'labels')
    image_out_dir = os.path.join(output_dir, subset, 'images')
    os.makedirs(label_dir, exist_ok=True)
    os.makedirs(image_out_dir, exist_ok=True)

    annos = defaultdict(list)
    for anno in coco["annotations"]:
        annos[anno["image_id"]].append(anno)

    print(f"🚀 正在转换 {subset} 数据集...")

    # 记录每个新文件名前缀的计数，避免重名
    counters = defaultdict(int)

    for image_id, img in tqdm(images.items()):
        file_name = img["file_name"]  # 可能含子目录，如 "book1/chapter1/page01.jpg"
        width = img["width"]
        height = img["height"]
        src_img_path = os.path.join(image_dir, file_name)

        if not os.path.exists(src_img_path):
            print(f"⚠️ 找不到图片: {src_img_path}")
            continue

        # 用下划线替换路径分隔符，得到新文件名前缀
        file_stem = Path(file_name).with_suffix('').as_posix().replace('/', '_').replace('\\', '_')

        # 计数+1
        counters[file_stem] += 1

        # 新文件名：路径拼接名 + _ + 4位序号 + 原后缀
        new_name = f"{file_stem}_{counters[file_stem]:04d}{Path(file_name).suffix}"

        # 拷贝图片到扁平目录并重命名
        dst_img_path = os.path.join(image_out_dir, new_name)
        shutil.copy(src_img_path, dst_img_path)

        # 写标签文件，名字对应图片
        label_file = os.path.join(label_dir, f"{Path(new_name).stem}.txt")
        with open(label_file, 'w') as f:
            for anno in annos.get(image_id, []):
                if anno.get("iscrowd", 0) == 1:
                    continue
                cat_id = anno["category_id"]
                bbox = anno["bbox"]  # [x, y, w, h]
                x_center = (bbox[0] + bbox[2] / 2) / width
                y_center = (bbox[1] + bbox[3] / 2) / height
                w = bbox[2] / width
                h = bbox[3] / height
                f.write(f"{categories[cat_id]} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}\n")

    return [cat["name"] for cat in coco["categories"]]

def create_yaml_file(save_path, class_names):
    yaml_path = os.path.join(save_path, 'data.yaml')
    with open(yaml_path, 'w', encoding='utf-8') as f:
        f.write(f"train: {save_path}/train/images\n")
        f.write(f"val: {save_path}/val/images\n")
        f.write(f"test: {save_path}/test/images\n\n")
        f.write(f"nc: {len(class_names)}\n")
        f.write(f"names: {class_names}\n")

def main():
    # === 修改这里 ===
    base_dir = ''  # 你的根目录路径
    json_dir = os.path.join(base_dir, 'text')      # 包含 train/val/test 的 json 文件夹
    image_dir = os.path.join(base_dir, 'images')   # 所有图片所在根目录（含子目录）
    output_dir = os.path.join(base_dir, 'yolo_format')

    subsets = {
        'train': 'train_text.json',
        'val': 'val_text.json',
        'test': 'test_text.json',
    }

    class_names = None
    for subset, json_file in subsets.items():
        json_path = os.path.join(json_dir, json_file)
        names = convert_coco_to_yolo(json_path, image_dir, output_dir, subset)
        if class_names is None:
            class_names = names

    create_yaml_file(output_dir, class_names)
    print("\n✅ 完成转换！YOLO 数据集路径：", output_dir)
    print("📄 data.yaml 已生成，可用于训练！")

if __name__ == "__main__":
    main()


In [20]:
import os
import cv2
import random
from pathlib import Path
import matplotlib.pyplot as plt

def show_yolo_labels(image_dir, label_dir, class_names=None, sample_num=2):
    image_paths = list(Path(image_dir).glob("*.jpg")) + list(Path(image_dir).glob("*.png"))
    sampled_images = random.sample(image_paths, sample_num)

    for img_path in sampled_images:
        label_path = Path(label_dir) / img_path.with_suffix('.txt').name

        if not label_path.exists():
            print(f"⚠️ 没有找到标签: {label_path}")
            continue

        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]

        with open(label_path, 'r') as f:
            lines = f.readlines()

        for line in lines:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls_id, x, y, bw, bh = map(float, parts)
            x1 = int((x - bw / 2) * w)
            y1 = int((y - bh / 2) * h)
            x2 = int((x + bw / 2) * w)
            y2 = int((y + bh / 2) * h)

            color = (0, 255, 0)
            label = str(int(cls_id)) if not class_names else class_names[int(cls_id)]
            cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
            cv2.putText(img, label, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        plt.figure(figsize=(10, 6))
        plt.imshow(img)
        plt.title(f"{img_path.name}")
        plt.axis('off')
        plt.show()

# === 修改路径 ===
image_dir = 'yolo_format/val/images'
label_dir = 'yolo_format/val/labels'
class_names = None  # 或者 ['cat', 'dog', ...]，可从 data.yaml 读取

show_yolo_labels(image_dir, label_dir, class_names)


🚀 正在转换 train 数据集...


100%|██████████| 8145/8145 [00:14<00:00, 552.84it/s]


🚀 正在转换 val 数据集...


100%|██████████| 410/410 [00:00<00:00, 620.08it/s]


🚀 正在转换 test 数据集...


100%|██████████| 2047/2047 [00:03<00:00, 613.09it/s]


✅ 完成转换！YOLO 数据集路径： yolo_format
📄 data.yaml 已生成，可用于训练！
